# Pipeline Integrado de Obtención, Inyección y Limpieza de Datos

**Proyecto:** Smart Kitchen Intelligence (SKI)  
**Objetivo:** Ejecutar el pipeline completo de preparación de datos

## Flujo del Pipeline

```
1. extract_patterns.py
   ↓
2. simulation.py → movements_raw.csv
   ↓
3. ingestion.py (NUEVA: OpenFoodFacts) → catalog_raw.csv
   ↓
4. anomaly_injection.py → movements_with_anomalies.csv
   ↓
5. preprocessing.py (MEJORADO) → inventory_v1.csv (LIMPIO)
   ↓
6. Análisis exploratorio → Tableau
```

## Cambios Realizados

### ✅ Fase 1: Actualización de Ingestion
- **Antes:** USDA API (legacy)
- **Después:** OpenFoodFacts API (actual)
- **Beneficio:** Mejor cobertura nutricional y Nutriscore nativo

### ✅ Fase 2: Inyección de Anomalías
- Nuevo módulo: `src/anomaly_injection.py`
- Simula datos reales "sucios"
- Anomalías inyectadas:
  - 15% valores nulos
  - 5% duplicados
  - 8% outliers
  - 3% inconsistencias lógicas
  - 5% errores de tipo

### ✅ Fase 3: Limpieza Robusta
- Mejorado: `src/preprocessing.py`
- Detección automática de anomalías
- Bitácora detallada de transformaciones
- Validación QA antes/después


## 1. CONFIGURACIÓN INICIAL

In [ ]:
import sys
import os
from pathlib import Path

# Agregar src al path
sys.path.insert(0, str(Path.cwd() / 'src'))

# Crear directorios necesarios
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/interim', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

print("✅ Directorios configurados")

## 2. GENERAR DATOS BASE (Simulación)

Ejecutar `simulation.py` para generar `movements_raw.csv` con transacciones realistas.

In [ ]:
import subprocess

print("🚀 Ejecutando simulación de movimientos...")
result = subprocess.run([sys.executable, 'src/simulation.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("❌ Error:", result.stderr)

## 3. OBTENER CATÁLOGO NUTRICIONAL (Nueva: OpenFoodFacts)

Ejecutar `ingestion.py` para obtener datos nutricionales desde OpenFoodFacts API.

In [ ]:
print("🌐 Obteniendo catálogo desde OpenFoodFacts API...")
result = subprocess.run([sys.executable, 'src/ingestion.py'], capture_output=True, text=True, timeout=300)
print(result.stdout)
if result.returncode != 0:
    print("⚠️ Advertencia:", result.stderr)

## 4. INYECTAR ANOMALÍAS REALISTAS

Crear versión "sucia" del dataset para simular datos del mundo real con problemas de calidad.

In [ ]:
print("⚠️ Inyectando anomalías realistas en el dataset...")
result = subprocess.run([
    sys.executable, 'src/anomaly_injection.py',
    '--input', 'data/raw/movements_raw.csv',
    '--output', 'data/interim/movements_with_anomalies.csv',
    '--seed', '42',
    '--null_ratio', '0.15'
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("❌ Error:", result.stderr)

## 5. VERIFICAR ANOMALÍAS INYECTADAS

In [ ]:
import pandas as pd
import json

# Cargar bitácora de inyección
with open('data/interim/movements_with_anomalies_injection_log.json', 'r') as f:
    injection_log = json.load(f)

print("📋 ANOMALÍAS INYECTADAS:")
print(json.dumps(injection_log['anomalies'], indent=2))

# Cargar dataset con anomalías
df_anomalous = pd.read_csv('data/interim/movements_with_anomalies.csv')
print(f"\n📊 Dataset con anomalías: {len(df_anomalous)} registros")
print(f"\n🔍 Nulos por columna:")
print(df_anomalous.isna().sum())

## 6. LIMPIAR Y CORREGIR ANOMALÍAS

Ejecutar `preprocessing.py` para detectar y corregir todas las anomalías.
Genera bitácora de transformaciones y validación QA.

In [ ]:
print("🧹 Ejecutando pipeline de limpieza y corrección...")
result = subprocess.run([
    sys.executable, 'src/preprocessing.py',
    '--input', 'data/interim/movements_with_anomalies.csv',
    '--catalog', 'data/raw/catalog_raw.csv',
    '--output', 'data/processed/inventory_v1.csv'
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("❌ Error:", result.stderr)

## 7. ANÁLISIS DE BITÁCORA DE LIMPIEZA

In [ ]:
# Cargar bitácora de limpieza
with open('data/processed/inventory_v1_cleaning_log.json', 'r') as f:
    cleaning_log = json.load(f)

print("📋 BITÁCORA DE LIMPIEZA Y CORRECCIÓN:")
print(json.dumps(cleaning_log, indent=2)[:2000] + "...")

print("\n📊 Resumen:")
print(f"Registros iniciales: {cleaning_log['initial_record_count']}")
print(f"Registros finales: {cleaning_log['final_record_count']}")
print(f"Registros removidos: {cleaning_log['initial_record_count'] - cleaning_log['final_record_count']}")

## 8. VALIDACIÓN DE DATOS LIMPIOS

In [ ]:
# Cargar dataset limpio
df_clean = pd.read_csv('data/processed/inventory_v1.csv')

print(f"✅ Dataset limpio cargado: {len(df_clean)} registros")
print(f"\n📊 Dimensiones:")
print(f"  Filas: {len(df_clean)}")
print(f"  Columnas: {len(df_clean.columns)}")

print(f"\n🔍 Nulos después de limpieza:")
null_counts = df_clean.isna().sum()
if null_counts.sum() == 0:
    print("  ✅ No hay valores nulos")
else:
    print(null_counts[null_counts > 0])

print(f"\n📋 Primeras 5 filas:")
print(df_clean.head())

print(f"\n📈 Resumen estadístico:")
print(df_clean.describe())

## 9. COMPARACIÓN ANTES vs DESPUÉS

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Comparación: Dataset Sucio vs Limpio', fontsize=16, fontweight='bold')

# 1. Cantidad de registros
ax = axes[0, 0]
registros = [len(df_anomalous), len(df_clean)]
labels = ['Con Anomalías', 'Limpio']
ax.bar(labels, registros, color=['#ff6b6b', '#51cf66'])
ax.set_ylabel('Cantidad de registros')
ax.set_title('Registros antes y después')
for i, v in enumerate(registros):
    ax.text(i, v + 100, str(v), ha='center', fontweight='bold')

# 2. Nulos
ax = axes[0, 1]
nulls_before = df_anomalous.isna().sum().sum()
nulls_after = df_clean.isna().sum().sum()
nulls_data = [nulls_before, nulls_after]
ax.bar(labels, nulls_data, color=['#ff6b6b', '#51cf66'])
ax.set_ylabel('Cantidad de valores nulos')
ax.set_title('Valores nulos antes y después')
for i, v in enumerate(nulls_data):
    ax.text(i, v + 100, str(int(v)), ha='center', fontweight='bold')

# 3. % de nulos por columna (antes)
ax = axes[1, 0]
null_pct_before = (df_anomalous.isna().sum() / len(df_anomalous) * 100).sort_values(ascending=False)[:5]
ax.barh(range(len(null_pct_before)), null_pct_before.values, color='#ff6b6b')
ax.set_yticks(range(len(null_pct_before)))
ax.set_yticklabels(null_pct_before.index)
ax.set_xlabel('% de nulos')
ax.set_title('Top 5 columnas con más nulos (ANTES)')

# 4. % de nulos por columna (después)
ax = axes[1, 1]
null_pct_after = (df_clean.isna().sum() / len(df_clean) * 100).sort_values(ascending=False)[:5]
if len(null_pct_after) > 0 and null_pct_after.max() > 0:
    ax.barh(range(len(null_pct_after)), null_pct_after.values, color='#51cf66')
    ax.set_yticks(range(len(null_pct_after)))
    ax.set_yticklabels(null_pct_after.index)
ax.set_xlabel('% de nulos')
ax.set_title('Top 5 columnas con más nulos (DESPUÉS)')
ax.set_xlim(0, 30)

plt.tight_layout()
plt.savefig('outputs/data_cleaning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Gráfica guardada en: outputs/data_cleaning_comparison.png")

## 10. GENERACIÓN DE REPORTE DE CALIDAD DE DATOS

In [ ]:
# Crear reporte de QA
qa_report = {
    "ejecutado_el": pd.Timestamp.now().isoformat(),
    "resumen_general": {
        "dataset_limpio": "data/processed/inventory_v1.csv",
        "registros_totales": len(df_clean),
        "columnas": len(df_clean.columns),
        "porcentaje_completitud": float((1 - (df_clean.isna().sum().sum() / (len(df_clean) * len(df_clean.columns)))) * 100)
    },
    "anomalias_removidas": {
        "registros_removidos": cleaning_log['initial_record_count'] - cleaning_log['final_record_count'],
        "duplicados": int(cleaning_log['anomalies_detected']['duplicates']),
        "inconsistencias_logicas": len(cleaning_log['anomalies_detected']['logical_errors'])
    },
    "integridad_datos": {
        "cantidad_nulos_totales": int(df_clean.isna().sum().sum()),
        "columnas_sin_nulos": int((df_clean.isna().sum() == 0).sum()),
        "datetime_range": {
            "inicio": str(df_clean['timestamp'].min()) if 'timestamp' in df_clean.columns else None,
            "fin": str(df_clean['timestamp'].max()) if 'timestamp' in df_clean.columns else None
        }
    },
    "validacion_tipos": {
        "all_quantity_numeric": bool(pd.api.types.is_numeric_dtype(df_clean['quantity'])),
        "all_timestamps_valid": bool(pd.api.types.is_datetime64_any_dtype(df_clean['timestamp']))
    }
}

# Guardar reporte
os.makedirs('docs', exist_ok=True)
with open('docs/data_quality_report.json', 'w') as f:
    json.dump(qa_report, f, indent=2, default=str)

print("✅ Reporte de QA guardado en: docs/data_quality_report.json")
print("\n📊 RESUMEN DE CALIDAD:")
print(json.dumps(qa_report, indent=2, default=str))

## ✅ PIPELINE COMPLETADO

### Archivos Generados

```
data/raw/
  ├── movements_raw.csv                    (original de simulación)
  ├── catalog_raw.csv                      (OpenFoodFacts)
  └── instacart_patterns.json              (patrones de referencia)

data/interim/
  ├── movements_with_anomalies.csv         (con anomalías inyectadas)
  └── movements_with_anomalies_injection_log.json

data/processed/
  ├── inventory_v1.csv                     (LIMPIO Y FINAL)
  └── inventory_v1_cleaning_log.json       (bitácora de transformaciones)

docs/
  └── data_quality_report.json             (reporte QA)

outputs/
  └── data_cleaning_comparison.png         (visualización antes/después)
```

### Próximos Pasos
1. ✅ **Ingestion.py** - Ahora obtiene datos de OpenFoodFacts
2. ✅ **Anomaly Injection** - Inyecta problemas realistas
3. ✅ **Preprocessing** - Detecta y limpia con bitácora
4. 📊 **Análisis Exploratorio** - Ver `01_perfilado.ipynb`
5. 📈 **Dashboard Tableau** - Usar `inventory_v1.csv`
